# NER Entity Cleaning & Review Export — Stage 1

Cleans and ranks named entities from all three NER CSVs and exports a three-sheet
review workbook (one sheet per type: PER / ORG / LOC) for manual curation.

Selection uses a **per-type quota** so persons are not crowded out by organisations.
Entities already in `entity_review_edited.xlsx` are pre-filled with your previous
`keep`, `canonical`, and corrected `type` values so you only need to review new rows.

**Sources and columns used:**
| Arena | File | Sub-unit col |
|---|---|---|
| news | `news/analysis/df_with_NER.csv` | `outlet` |
| talkshows | `subtitles/analysis/subs_with_NER.csv` | `program` |
| kamer | `tweede_kamer/analysis/Tweede_Kamer_with_NER.csv` | `type` |

In [1]:
import ast
import re
import pathlib
import pandas as pd

NB_DIR = pathlib.Path(".").resolve()

# ============================================================
# PARAMETERS — edit before running
# ============================================================

SOURCES = {
    "news": {
        "path":       "../../news/analysis/df_with_NER.csv",
        "subunit_col": "outlet",
    },
    "talkshows": {
        "path":       "../../subtitles/analysis/subs_with_NER.csv",
        "subunit_col": "program",
    },
    "kamer": {
        "path":       "../../tweede_kamer/analysis/Tweede_Kamer_with_NER.csv",
        "subunit_col": "type",
    },
}

ENTITY_COLS = {
    "persons":   "PER",
    "orgs":      "ORG",
    "countries": "LOC",
}

EXISTING_REVIEW = "entity_review_edited.xlsx"   # pre-fill source; skipped if absent
OUTPUT_XLSX     = "entity_review.xlsx"

PER_TYPE_TOP_N = 300   # top N entities kept per type (PER / ORG / LOC)
MIN_DOC_FREQ   = 2     # drop entities appearing in fewer than this many documents
# ============================================================

## 1. Helpers

In [2]:
def parse_entity_list(cell):
    """Parse a list-valued cell → list of raw entity strings."""
    if pd.isna(cell):
        return []
    s = str(cell).strip()
    if s in ("", "[]", "nan"):
        return []
    try:
        result = ast.literal_eval(s)
        if isinstance(result, list):
            return [str(x) for x in result]
        return [str(result)]
    except (ValueError, SyntaxError):
        s = re.sub(r"^[\[\(]|[\]\)]$", "", s)
        return [x.strip().strip("'\"" ) for x in s.split(",") if x.strip()]


def normalise(s):
    """Collapse internal whitespace and strip edges."""
    return re.sub(r"\s+", " ", str(s).strip())

## 2. Load sources

In [3]:
frames = {}

for arena, cfg in SOURCES.items():
    abs_path = (NB_DIR / cfg["path"]).resolve()
    if not abs_path.exists():
        print(f"[SKIP] {arena}: not found at {abs_path}")
        continue
    df = pd.read_csv(abs_path)
    missing = [c for c in ENTITY_COLS if c not in df.columns]
    if missing:
        print(f"[SKIP] {arena}: missing columns {missing}")
        continue
    frames[arena] = df
    print(f"[OK] {arena}: {len(df):,} rows")

if not frames:
    raise RuntimeError("No source files loaded.")

[OK] news: 13,209 rows
[OK] talkshows: 495 rows
[OK] kamer: 844 rows


## 3. Parse → long format → noise filter

In [4]:
records = []
for arena, cfg in SOURCES.items():
    if arena not in frames:
        continue
    df = frames[arena]
    for row_idx, row in df.iterrows():
        doc_id = f"{arena}:{row_idx}"
        arena_label = arena
        for col, etype in ENTITY_COLS.items():
            for raw in parse_entity_list(row.get(col)):
                norm = normalise(raw)
                records.append({
                    "doc_id": doc_id,
                    "arena":  arena_label,
                    "norm":   norm,
                    "type":   etype,
                })

long_df = pd.DataFrame(records)
print(f"Raw mentions: {len(long_df):,}")

before = len(long_df)
long_df = long_df[
    (long_df["norm"].str.len() > 1) &
    (~long_df["norm"].str.fullmatch(r"\d+"))
].copy()
print(f"After noise filter: {len(long_df):,} (dropped {before - len(long_df):,})")

long_df["key"] = long_df["norm"].str.lower()

Raw mentions: 236,790
After noise filter: 236,264 (dropped 526)


## 4. Aggregate to frequency table

In [5]:
display_label = (
    long_df.groupby("key")["norm"]
    .agg(lambda x: x.value_counts().index[0])
    .rename("entity")
)
doc_freq = (
    long_df.groupby("key")["doc_id"]
    .nunique()
    .rename("doc_freq")
)
sources_col = (
    long_df.groupby("key")["arena"]
    .agg(lambda x: ", ".join(sorted(x.unique())))
    .rename("sources")
)

type_doc    = long_df[["key", "doc_id", "type"]].drop_duplicates()
type_counts = type_doc.groupby(["key", "type"]).size().reset_index(name="n")
dominant_type = (
    type_counts
    .loc[type_counts.groupby("key")["n"].idxmax()]
    .set_index("key")["type"]
)
multi_type_flag = (
    type_counts.groupby("key")["type"]
    .count()
    .gt(1)
    .map({True: "yes", False: ""})
    .rename("multi_type")
)

freq_df = pd.concat(
    [display_label, dominant_type, doc_freq, sources_col, multi_type_flag],
    axis=1,
).reset_index(drop=True)

print(f"Unique entities: {len(freq_df):,}")

Unique entities: 81,248


## 5. Per-type quota selection

Apply `min_doc_freq` floor, then take the top `per_type_top_n` within each type
ranked by document frequency. Each entity appears on the sheet matching its
dominant type.

In [6]:
freq_filtered = freq_df[freq_df["doc_freq"] >= MIN_DOC_FREQ].copy()
print(f"After min_doc_freq={MIN_DOC_FREQ}: {len(freq_filtered):,} entities")
print(freq_filtered["type"].value_counts().rename("pool size").to_string())
print()

parts = []
for etype in ["PER", "ORG", "LOC"]:
    subset = (
        freq_filtered[freq_filtered["type"] == etype]
        .sort_values("doc_freq", ascending=False)
        .head(PER_TYPE_TOP_N)
    )
    parts.append(subset)
    print(f"  {etype}: selected {len(subset)} of {(freq_filtered['type']==etype).sum()} "
          f"(top-{PER_TYPE_TOP_N}, lowest doc_freq={int(subset['doc_freq'].iloc[-1]) if len(subset) else '-'})")

selected = pd.concat(parts, ignore_index=True)

After min_doc_freq=2: 20,977 entities
type
PER    10619
ORG     6930
LOC     3428

  PER: selected 300 of 10619 (top-300, lowest doc_freq=22)
  ORG: selected 300 of 6930 (top-300, lowest doc_freq=29)
  LOC: selected 300 of 3428 (top-300, lowest doc_freq=20)


## 6. Pre-fill from existing curated review

For entities already in `entity_review_edited.xlsx`: copy `keep`, `canonical`, and
corrected `type`. New entities get `keep="yes"` and `canonical=entity`.

Sheet placement uses the **final** type (post-correction), so a PER entity you
reclassified to ORG in the previous round will appear on the ORG sheet.

In [7]:
existing_path = (NB_DIR / EXISTING_REVIEW).resolve()

if existing_path.exists():
    old = pd.read_excel(existing_path)
    # Build lookup keyed on normalised entity string (lowercase)
    old_lookup = {
        normalise(row["entity"]).lower(): {
            "keep":      str(row["keep"]).strip(),
            "canonical": normalise(row["canonical"]),
            "type":      str(row["type"]).strip(),
        }
        for _, row in old.iterrows()
    }
    print(f"Pre-fill source: {existing_path.name}  ({len(old)} rows)")
else:
    old_lookup = {}
    print(f"'{EXISTING_REVIEW}' not found — all rows will be new.")

def apply_prefill(row):
    key = normalise(row["entity"]).lower()
    if key in old_lookup:
        old = old_lookup[key]
        row["keep"]      = old["keep"]
        row["canonical"] = old["canonical"]
        row["type"]      = old["type"]   # corrected type overrides dominant
    else:
        row["keep"]      = "yes"
        row["canonical"] = row["entity"]
    return row

selected = selected.apply(apply_prefill, axis=1)

prefilled = selected["keep"].notna().sum()
from_old  = sum(1 for _, r in selected.iterrows()
                if normalise(r["entity"]).lower() in old_lookup)
print(f"Entities with pre-filled values (from old review): {from_old}")
print(f"New entities (keep=yes, canonical=entity):          {len(selected) - from_old}")

'entity_review_edited.xlsx' not found — all rows will be new.
Entities with pre-filled values (from old review): 0
New entities (keep=yes, canonical=entity):          900


## 7. Export — one sheet per type

Sheet placement is based on the final (possibly corrected) type.
Within each sheet, rows are sorted by doc_freq descending.

In [8]:
OUTPUT_COLS = ["entity", "type", "doc_freq", "sources", "multi_type", "keep", "canonical"]

output_path = NB_DIR / OUTPUT_XLSX
with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    for sheet in ["PER", "ORG", "LOC"]:
        sheet_df = (
            selected[selected["type"] == sheet]
            .sort_values("doc_freq", ascending=False)
            [OUTPUT_COLS]
            .reset_index(drop=True)
        )
        sheet_df.to_excel(writer, sheet_name=sheet, index=False)

print(f"Saved → {output_path}")

Saved → C:\Users\joly-\Github\HUMAN\Comparative_visualisations\Entity-NER\entity_review.xlsx


## 8. Summary

In [9]:
print("=" * 55)
print("SUMMARY")
print(f"  Raw mentions parsed:    {len(records):>8,}")
print(f"  After noise filter:     {len(long_df):>8,}")
print(f"  Unique entities:        {len(freq_df):>8,}")
print(f"  After min_doc_freq={MIN_DOC_FREQ}:  {len(freq_filtered):>8,}")
print()
print(f"  SHEETS (final type after pre-fill corrections):")
for sheet in ["PER", "ORG", "LOC"]:
    n      = (selected["type"] == sheet).sum()
    n_old  = sum(1 for _, r in selected[selected["type"]==sheet].iterrows()
                 if normalise(r["entity"]).lower() in old_lookup)
    n_new  = n - n_old
    cutoff = int(selected[selected["type"]==sheet]["doc_freq"].min()) if n else "-"
    print(f"    {sheet}: {n:>4} rows  "
          f"({n_old} pre-filled, {n_new} new)  "
          f"lowest doc_freq={cutoff}")
print("=" * 55)

SUMMARY
  Raw mentions parsed:     236,790
  After noise filter:      236,264
  Unique entities:          81,248
  After min_doc_freq=2:    20,977

  SHEETS (final type after pre-fill corrections):
    PER:  300 rows  (0 pre-filled, 300 new)  lowest doc_freq=22
    ORG:  300 rows  (0 pre-filled, 300 new)  lowest doc_freq=29
    LOC:  300 rows  (0 pre-filled, 300 new)  lowest doc_freq=20
